# Scénario 11b : anticiper les codifications de septembre, cible binaire (sain ou en retard)

**Variante du [`11a`](11a_anticipation_pay1.ipynb), un seul changement : la cible est binaire.** Au lieu de quatre familles, le modèle prédit si la codification de septembre est **saine (-2, -1 ou 0)** ou **en retard (2 et plus)** : une simplification, pour alléger le travail des modèles sur la question qui compte (sortir du retard, y rester, y entrer). La codification exacte (-2, -1 ou 0) n'est plus prédite. Le reste est identique au `11a` (correctifs `10` puis règle fixe des retards, mois empilés, boucle, enregistrement) ; les modèles sont enregistrés dans `modeles/11b/`.

Journal : [`tableau_resultats.md`](tableau_resultats.md). Essai **bonus et indépendant** des autres branches de machine learning du projet.

**Objet** : entraîner des modèles à prédire la codification de septembre (`PAY_1`) en deux classes (**sain ou en retard**), sur les clients dont `PAY_1` n'est pas un 1, et **enregistrer les cinq modèles réglés** dans `modeles/11b/`. Le notebook `11b` les chargera pour estimer ce que cachent les `PAY_1` = 1 (colonne estimée et confiance) ; le choix du modèle se fera selon les résultats et les critères habituels. **Pas de réapprentissage supplémentaire** : chaque modèle enregistré est appris sur tout le train avec son meilleur réglage ; le test de septembre n'est pas lu.

**Données** : `data/creditcard_pret_ingestion.csv` (niveau 0), **sans `dpnm`, sans `ID`, sans données personnelles** (`SEX`, `EDUCATION`, `MARRIAGE`, `AGE`). Avant tout, les codifications d'avril à août sont corrigées selon la **codification anticipée** (la codification du mois M porte sur la facture du mois, réglée le mois suivant), dans l'ordre :
1. **corrections des codifications 1 d'avril à août** (`10`) : règle du solde du mois suivant nul (codification saine antérieure), règle de l'encours nul, règle des impayés, décisions de l'utilisateur ;
2. **règle fixe des retards** (passage 18 du journal, abandonnée depuis dans le projet) : une facture du mois d'au moins 100 NT$ sans aucun paiement le mois suivant vaut un retard (2, ou niveau du mois précédent + 1).

`PAY_1` n'est pas corrigé : c'est la cible.

**Méthode** : celle du scénario au meilleur score optimisé, le **`01f`** (mois d'avril à septembre empilés, colonnes rangées par ancienneté, montants et plafond au logarithme signé, codifications regroupées, mêmes 5 plis par client notés sur septembre, score en train calculé aussi sur septembre, boucle d'optimisation, surapprentissage limité à 7 % relatif, alerte si l'écart-type des plis dépasse 2 points, régression logistique avec choix de pénalité L1, profondeur des arbres plafonnée à 30). **Départ de la boucle : les meilleurs réglages du `01f`.**

## 1. Données

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")
df = df.drop(columns="dpnm")   # jamais utilisé dans ce notebook
PAY = [f"PAY_{n}" for n in range(1, 7)]
MOIS = {1: "septembre", 2: "août", 3: "juillet", 4: "juin", 5: "mai", 6: "avril"}
print(f"Clients du niveau 0 : {len(df)} ; colonnes : {df.shape[1]} (dpnm retirée) ; PAY_1 = 1 : {int((df['PAY_1'] == 1).sum())}")

# Codifications 1 d'avril à août : leurs clients ont tous PAY_1 = 1, ils restent donc hors du train et du test
anciens_1 = (df[PAY[1:]] == 1).any(axis=1)
assert (df.loc[anciens_1, "PAY_1"] == 1).all()
print(f"Clients avec une codification 1 d'avril à août : {int(anciens_1.sum())}, tous codifiés 1 aussi en septembre")

## 1 bis. Corrections des codifications d'avril à août (`10` puis règle fixe des retards)

Les fonctions reprennent celles du notebook `10`, puis la règle fixe des retards. Après les corrections du `10`, il ne doit plus rester aucune codification 1 d'avril à août ; la règle fixe des retards s'applique ensuite. Le tableau compte les codifications modifiées par chaque étape.

In [ ]:
SEUIL = 100   # NT$ : règle fixe des retards (09)
avant_corrections = df[PAY].copy()
journal = {}

def saine_anterieure(ligne, n):
    for k in range(n + 1, 7):
        if ligne[f"PAY_{k}"] <= 0:
            return ligne[f"PAY_{k}"]
    return None

# 10, étape 0 : PAY_n = 1 et solde du mois suivant nul ou créditeur -> codification saine antérieure
nb = 0
for n in range(6, 1, -1):
    for k in df.index[(df[f"PAY_{n}"] == 1) & (df[f"BILL_AMT{n - 1}"] <= 0)]:
        v = saine_anterieure(df.loc[k], n)
        if v is not None:
            df.at[k, f"PAY_{n}"] = v; nb += 1
journal["10 : solde du mois suivant nul (codification saine antérieure)"] = nb

# 10, étape 1 : encours nul (pas de facture du mois, mois précédent sain) -> codification du mois précédent
nb = 0
for n in range(5, 1, -1):
    regle = (df[f"PAY_{n}"] == 1) & (df[f"BILL_AMT{n}"] <= 0) & (df[f"PAY_{n + 1}"] <= 0)
    df.loc[regle, f"PAY_{n}"] = df.loc[regle, f"PAY_{n + 1}"]; nb += int(regle.sum())
journal["10 : encours nul"] = nb

# 10, étape 2 : impayés sur une codification 1 (facture >= 100 NT$, rien payé le mois suivant) -> retard
nb = 0
for n in range(6, 1, -1):
    regle = (df[f"PAY_{n}"] == 1) & (df[f"BILL_AMT{n}"] >= SEUIL) & (df[f"PAY_AMT{n - 1}"] == 0)
    avant = df[f"PAY_{n + 1}"] if n < 6 else pd.Series(np.nan, index=df.index)
    df.loc[regle, f"PAY_{n}"] = pd.Series(np.where(avant >= 2, avant + 1, 2), index=df.index)[regle]; nb += int(regle.sum())
journal["10 : impayés sur une codification 1"] = nb

# 10, étape 3 : décisions de l'utilisateur (codification anticipée)
DECISIONS = {6783: {4: 0, 3: 0, 2: 0}, 11498: {4: 2, 3: 2, 2: 2}, 892: {3: 0, 2: 0}, 1967: {3: 0, 2: 0},
             5645: {2: 0}, 19195: {2: 2}, 25176: {2: 2}}
ids_df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv", usecols=["ID"])["ID"]
nb = 0
for id_client, valeurs in DECISIONS.items():
    ligne = ids_df.index[ids_df == id_client]
    assert len(ligne) == 1, id_client
    for n, v in valeurs.items():
        df.loc[ligne, f"PAY_{n}"] = v; nb += 1
journal["10 : décisions de l'utilisateur"] = nb
assert int((df[PAY[1:]] == 1).sum().sum()) == 0, "il reste des codifications 1 d'avril à août"

# 09 : règle fixe des retards, d'avril à août
nb = 0
for n in range(6, 1, -1):
    regle = (df[f"PAY_{n}"] <= 1) & (df[f"BILL_AMT{n}"] >= SEUIL) & (df[f"PAY_AMT{n - 1}"] == 0)
    avant = df[f"PAY_{n + 1}"] if n < 6 else pd.Series(np.nan, index=df.index)
    df.loc[regle, f"PAY_{n}"] = pd.Series(np.where(avant >= 2, avant + 1, 2), index=df.index)[regle]; nb += int(regle.sum())
journal["09 : règle fixe des retards"] = nb
assert (df["PAY_1"] == avant_corrections["PAY_1"]).all(), "PAY_1 ne doit pas être corrigé"

display(pd.DataFrame({"Codifications modifiées": journal}))
print("Codifications modifiées par mois (avril à août) :",
      {MOIS[n]: int((df[f"PAY_{n}"] != avant_corrections[f"PAY_{n}"]).sum()) for n in range(6, 1, -1)})

## 2. Le découpage de septembre (identique au 01e)

Les clients dont `PAY_1` n'est pas un 1, coupés en 80/20 (train / test, `stratify` sur `PAY_1`, `random_state=42`), exactement comme dans les notebooks 01 à 01e : mêmes clients dans le train, mêmes clients dans le test. Les clients du test ne servent à aucun mois de l'apprentissage. Les variables d'origine retenues sont les mêmes que dans le 01e (codifications d'avril à août, soldes, paiements, plafond ; ni `ID`, ni `dpnm`, ni données personnelles) ; elles sont rangées par mois dans la section 3.

In [ ]:
from sklearn.model_selection import train_test_split

# Variables : colonnes du dataset d'origine (niveau 0), sauf ID (identifiant), PAY_1 (la cible), dpnm et les données personnelles ; aucune colonne ajoutée
PERSONNELLES = ["SEX", "EDUCATION", "MARRIAGE", "AGE"]   # retirées : non pertinentes pour une codification de paiement
NUMERIQUES = ["LIMIT_BAL"] + [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)]
CATEGORIELLES = PAY[1:]
VARIABLES = CATEGORIELLES + NUMERIQUES
assert sorted(VARIABLES + ["ID", "PAY_1"] + PERSONNELLES) == sorted(df.columns)
mask_1 = df["PAY_1"] == 1
connus = df[~mask_1]
a_trancher = df[mask_1]

X_train, X_test, y_train, y_test = train_test_split(
    connus[VARIABLES], connus["PAY_1"], test_size=0.2, stratify=connus["PAY_1"], random_state=42)
CLASSES = sorted(connus["PAY_1"].unique())   # toutes les codifications de PAY_1, sauf 1

repartition = pd.DataFrame({
    "Train": y_train.value_counts(),
    "Test": y_test.value_counts(),
}).reindex(CLASSES).fillna(0).astype(int)
repartition.loc["Total"] = repartition.sum()
print(f"PAY_1 connus : {len(connus)} (train {len(X_train)}, test {len(X_test)}) ; PAY_1 = 1 à trancher : {len(a_trancher)}")
print(f"Variables ({len(VARIABLES)}) : {VARIABLES}")
display(repartition)

**Comment lire** : nombre de clients par codification de `PAY_1`, dans le train et dans le test. Les proportions sont les mêmes dans les deux (`stratify`). Le test n'est plus touché jusqu'au critère A.

## 3. Les mois empilés

Chaque exemple est un client à un mois donné, avec ce que la banque voyait à la fin de ce mois (colonnes relatives, choix 1 de l'introduction) ; la cible est la codification de ce mois, regroupée en -2, -1, 0, 2 et plus. Les codifications des mois précédents sont regroupées de la même façon. Le tableau donne le nombre d'exemples par mois et par codification.

In [ ]:
def regrouper(codifications):
    # 2 et plus -> 2 (retard) ; -2, -1, 0 (et 1) inchangés ; un mois absent (vide) reste vide
    return codifications.mask(codifications >= 2, 2)

def exemples_du_mois(clients, n):
    # Ce que la banque voyait à la fin du mois n : codifications des mois précédents (m1 = mois précédent),
    # soldes et paiements du mois (m0) et des mois précédents ; vide si le mois est antérieur à avril
    d = df.loc[clients]
    bloc = pd.DataFrame(index=d.index)
    for k in range(1, 6):
        bloc[f"codif_m{k}"] = regrouper(d[f"PAY_{n + k}"]) if n + k <= 6 else np.nan
    for k in range(0, 6):
        bloc[f"solde_m{k}"] = d[f"BILL_AMT{n + k}"] if n + k <= 6 else np.nan
        bloc[f"paiement_m{k}"] = d[f"PAY_AMT{n + k}"] if n + k <= 6 else np.nan
    bloc["LIMIT_BAL"] = d["LIMIT_BAL"]
    bloc["cible"] = regrouper(d[f"PAY_{n}"])
    bloc["mois"] = n
    bloc["client"] = d.index
    return bloc

MOIS_NOMS = {1: "septembre", 2: "août", 3: "juillet", 4: "juin", 5: "mai"}
clients_sept = X_train.index                  # clients du train de septembre (PAY_1 connu)
clients_1 = a_trancher.index                  # clients codifiés 1 en septembre : seulement leurs mois d'avril à août
emp = pd.concat([exemples_du_mois(clients_sept, 1)]
                + [exemples_du_mois(clients_sept.union(clients_1), n) for n in range(2, 6)], ignore_index=True)
emp = emp[emp["cible"] != 1].reset_index(drop=True)   # cibles codifiées 1 (avril à août) retirées
assert not emp["client"].isin(X_test.index).any(), "un client du test est dans les exemples"
assert (emp["mois"] == 1).sum() == len(X_train)

CATEGORIELLES = [f"codif_m{k}" for k in range(1, 6)]
NUMERIQUES = [f"solde_m{k}" for k in range(0, 6)] + [f"paiement_m{k}" for k in range(0, 6)] + ["LIMIT_BAL"]
VARIABLES = CATEGORIELLES + NUMERIQUES
X_emp, y_emp = emp[VARIABLES], emp["cible"].astype(int)
y_emp = y_emp.where(y_emp >= 2, 0)   # 11b : cible binaire, 0 = sain (-2, -1, 0), 2 = en retard (2 et plus)

# Septembre : mêmes variables que le 01e, dans le même ordre de clients
sept = np.flatnonzero(emp["mois"].to_numpy() == 1)
assert (emp.loc[sept, "codif_m1"].to_numpy() == regrouper(X_train["PAY_2"]).to_numpy()).all()

# Seuls les exemples de septembre ont une codification cinq mois avant (avril) : codif_m5 renseignée <=> septembre
assert (emp["codif_m5"].notna() == (emp["mois"] == 1)).all()

CLASSES = sorted(y_emp.unique())
tableau = pd.crosstab(emp["mois"].map(MOIS_NOMS).rename("Mois"), y_emp.rename("Codification du mois"), margins=True, margins_name="Total")
print(f"Exemples : {len(emp)} ({emp['client'].nunique()} clients) ; classes : {CLASSES}")
display(tableau.reindex(list(MOIS_NOMS.values()) + ["Total"]))

**Comment lire** : une ligne par mois, une colonne par codification du mois ; septembre ne compte que les clients du train, les autres mois comptent aussi les clients codifiés 1 en septembre. Les mois d'avril à août apportent la plupart des exemples, et en particulier des retards.

## 4. Le score optimisé : la moyenne des taux de bonne classe (sain, en retard)

Comme dans le `11a`, chaque classe pèse autant : la moyenne du taux de bonnes prédictions parmi les sains et parmi les retards (`balanced_accuracy`). Le pourcentage de bonnes prédictions global reste affiché dans les tableaux de la section 8.


In [ ]:
# ==============================================================================
# Score optimisé (choix validé) : "familles équilibrées" ; autre possibilité : "bonnes prédictions"
# ==============================================================================
SCORE = "familles équilibrées"
SURAPPRENTISSAGE_MAX = 7.0   # écart relatif train - validation, en %
VARIANCE_MAX = 2.0           # écart-type des 5 plis, en points
MAX_TOURS = 10               # tours de la boucle, par modèle

import time
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, make_scorer
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

FAMILLES = [0, 2]   # 11b : deux classes
NOMS_FAMILLES = {0: "sain (-2, -1, 0)", 2: "2 et plus"}

def famille_pay(codifications):
    # Codification -> famille : -2, -1, 0 gardés, 2 et plus réunis en 2
    v = np.asarray(codifications).ravel().astype(int)
    return np.where(v >= 2, 2, 0)   # 11b : sain (-2, -1, 0) -> 0, retard -> 2

def score_familles(vrai, predit):
    v, p = famille_pay(vrai), famille_pay(predit)
    return balanced_accuracy_score(v, p) if SCORE == "familles équilibrées" else (v == p).mean()

assert SCORE in ("familles équilibrées", "bonnes prédictions")
def score_septembre(estimateur, X, y):
    # Score sur les seuls exemples de septembre (codif_m5 renseignée) : en validation, ce sont tous les exemples ;
    # en train, le septembre des clients appris, pour que le contrôle du surapprentissage compare la même chose
    septembre = X["codif_m5"].notna().to_numpy()
    return score_familles(np.asarray(y)[septembre], estimateur.predict(X[septembre]))

scoreur = score_septembre

def bonnes_predictions(vrai, predit):
    # Pourcentage de bonnes prédictions (bonne famille), en tout et dans chaque famille
    v, p = famille_pay(vrai), famille_pay(predit)
    resultat = {"global": (v == p).mean() * 100}
    for f in FAMILLES:
        resultat[f] = (p[v == f] == f).mean() * 100
    return resultat

# Logarithme signé des soldes, des paiements et du plafond, comme dans 01b (un vide reste vide)
MONTANTS_DUS_PAYES = NUMERIQUES

def log_montants(X):
    X = X.copy()
    X[MONTANTS_DUS_PAYES] = np.sign(X[MONTANTS_DUS_PAYES]) * np.log1p(np.abs(X[MONTANTS_DUS_PAYES]))
    return X

def etape_log():
    return FunctionTransformer(log_montants)

def preprocesseur_lineaire():
    # Codifications : une colonne par valeur (un mois absent est une valeur à part) ;
    # nombres (déjà au logarithme) : un vide est remplacé par 0 et signalé par un indicateur, puis mise à l'échelle standard
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
        ("nombres", make_pipeline(SimpleImputer(strategy="constant", fill_value=0, add_indicator=True), StandardScaler()), NUMERIQUES),
    ])

def preprocesseur_arbres():
    # Catégories : une colonne par valeur ; nombres tels quels (déjà au logarithme ; les arbres et CatBoost traitent les vides)
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
    ], remainder="passthrough")

# Plis : les mêmes 5 plis que le 01e, par client ; chaque pli est noté sur le septembre de ses clients,
# et ces clients sont retirés de l'apprentissage du pli pour tous leurs mois
plis_sept = StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(sept, y_emp.to_numpy()[sept])
clients_emp = emp["client"].to_numpy()
cv = []
for _, val in plis_sept:
    clients_val = np.unique(clients_emp[sept[val]])
    apprentissage = np.flatnonzero(~np.isin(clients_emp, clients_val))
    cv.append((apprentissage, sept[val]))
print(f"Plis : apprentissage de {min(len(a) for a, _ in cv)} à {max(len(a) for a, _ in cv)} exemples (tous mois), validation de {min(len(v) for _, v in cv)} à {max(len(v) for _, v in cv)} clients (septembre)")
print(f"Score optimisé : {SCORE} ; surapprentissage max {SURAPPRENTISSAGE_MAX} % relatif ; alerte si écart-type des plis > {VARIANCE_MAX} points")

## 5. Les réglages explorés

Pour chaque modèle : les **échelles** (paramètres ordonnés, explorés par la boucle autour d'une valeur de départ : ici, **le meilleur réglage du 01f**) et les **choix** (paramètres sans ordre, tous essayés à chaque tour). Les autres paramètres gardent leur valeur par défaut, sauf ceux indiqués en commentaire.

- **Régression logistique** : force de la régularisation `C` (petit = forte régularisation) ; poids des classes (`class_weight` : aucun, ou équilibré pour compenser les codifications rares) ; **type de pénalité** (`l1_ratio` : 0 = L2, qui réduit tous les coefficients ; 1 = L1, qui peut **annuler** les variables inutiles, par exemple des mois trop anciens ; 0,5 = mélange des deux), avec le solveur `saga`, le seul qui accepte la pénalité L1 pour plusieurs classes.
- **Arbre de décision** : profondeur maximale (plafonnée à 30), taille minimale des feuilles (grande = règles plus générales) ; poids des classes ; critère de coupe (`gini` ou `entropy`).
- **RandomForest** : nombre d'arbres, profondeur maximale (plafonnée à 30, sans profondeur illimitée), taille minimale des feuilles ; part des variables essayées à chaque coupe (`max_features`) ; poids des classes.
- **MLP** : taille des couches cachées, régularisation `alpha` ; arrêt anticipé (`early_stopping`) pour éviter d'apprendre par cœur.
- **CatBoost** : réglé comme dans le machine learning de la deuxième itération (même pipeline que les arbres, codifications en une colonne par valeur ; tous les cœurs ; 1 000 itérations). La boucle explore la profondeur, le `learning_rate` et `l2_leaf_reg` ; poids des classes : `Balanced` ou `SqrtBalanced` (plus doux), toujours actifs, CatBoost refusant la valeur « aucun » passée explicitement.

In [ ]:
ESPACES = {
    "Régression logistique": {
        "modele": make_pipeline(etape_log(), preprocesseur_lineaire(), LogisticRegression(solver="saga", max_iter=5000)),   # saga : seul solveur qui accepte la pénalité L1 en multiclasse
        "n_jobs": -1,
        "echelles": {"logisticregression__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300]},
        "depart": {"logisticregression__C": 0.1},   # meilleur réglage du 01f
        "choix": {"logisticregression__class_weight": [None, "balanced"],
                  "logisticregression__l1_ratio": [0, 0.5, 1]},   # 0 : pénalité L2 (réduit les coefficients) ; 1 : L1 (peut en annuler) ; 0,5 : mélange
    },
    "Arbre de décision": {
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(), DecisionTreeClassifier(random_state=42)),
        "n_jobs": -1,
        "echelles": {"decisiontreeclassifier__max_depth": [3, 4, 5, 6, 8, 10, 12, 15, 20, 30],   # plafond à 30 (pas de profondeur illimitée)
                     "decisiontreeclassifier__min_samples_leaf": [1, 2, 5, 10, 20, 50, 100, 200]},
        "depart": {"decisiontreeclassifier__max_depth": 8, "decisiontreeclassifier__min_samples_leaf": 2},   # meilleur réglage du 01f
        "choix": {"decisiontreeclassifier__class_weight": [None, "balanced"],
                  "decisiontreeclassifier__criterion": ["gini", "entropy"]},
    },
    "RandomForest": {
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(), RandomForestClassifier(random_state=42, n_jobs=-1)),
        "n_jobs": -1,
        "echelles": {"randomforestclassifier__n_estimators": [100, 200, 300, 500, 1000],
                     "randomforestclassifier__max_depth": [6, 8, 10, 12, 15, 20, 25, 30],   # plafond à 30 (pas de profondeur illimitée)
                     "randomforestclassifier__min_samples_leaf": [1, 2, 3, 5, 10, 20, 50]},
        "depart": {"randomforestclassifier__n_estimators": 300, "randomforestclassifier__max_depth": 25, "randomforestclassifier__min_samples_leaf": 10},   # meilleur réglage du 01f
        "choix": {"randomforestclassifier__max_features": ["sqrt", 0.5],
                  "randomforestclassifier__class_weight": [None, "balanced_subsample"]},
    },
    "MLP": {
        "modele": make_pipeline(etape_log(), preprocesseur_lineaire(), MLPClassifier(max_iter=500, early_stopping=True, random_state=42)),
        "n_jobs": -1,
        "echelles": {"mlpclassifier__hidden_layer_sizes": [(32,), (64,), (100,), (128, 64), (256, 128), (256, 128, 64)],
                     "mlpclassifier__alpha": [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]},
        "depart": {"mlpclassifier__hidden_layer_sizes": (128, 64), "mlpclassifier__alpha": 1e-5},   # meilleur réglage du 01f
        "choix": {},
    },
    "CatBoost": {
        # Réglé comme dans le machine learning de la deuxième itération : même pipeline que les arbres (codifications en une colonne par valeur),
        # tous les cœurs, pas de fichiers écrits ; poids des classes toujours actifs (CatBoost refuse auto_class_weights=None passé explicitement)
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(),
                                CatBoostClassifier(loss_function="MultiClass", iterations=1000, random_seed=42, verbose=0,
                                                   thread_count=-1, allow_writing_files=False)),
        "n_jobs": -1,
        "echelles": {"catboostclassifier__depth": [2, 3, 4, 5, 6, 7, 8, 10],
                     "catboostclassifier__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.1, 0.2],
                     "catboostclassifier__l2_leaf_reg": [1, 3, 5, 10, 15, 30, 60]},
        "depart": {"catboostclassifier__depth": 7, "catboostclassifier__learning_rate": 0.02, "catboostclassifier__l2_leaf_reg": 5},   # meilleur réglage du 01f
        "choix": {"catboostclassifier__auto_class_weights": ["Balanced", "SqrtBalanced"]},
    },
}
for nom, e in ESPACES.items():
    for p, v in e["depart"].items():
        assert v in e["echelles"][p], (nom, p)
print("Modèles :", list(ESPACES))

## 6. La boucle d'optimisation

Pour chaque modèle, tour après tour : GridSearch sur les 5 plis par client (notés sur septembre) sur la grille courante, puis choix du meilleur réglage **parmi ceux qui surapprennent de 7 % relatif au plus** ; le score en train et le score en validation portent tous deux sur septembre. Une ligne à la fin de chaque tour (réglages essayés, réglages admis, meilleur admis, durée du tour), pour suivre l'avancement ; puis, pour chaque modèle, une ligne : son meilleur réglage admis, son score en validation croisée et la raison de l'arrêt de la boucle.

In [ ]:
def fenetre(echelle, position):
    # La valeur courante et ses deux voisines dans l'échelle
    return echelle[max(0, position - 1): position + 2]

def libelle(params):
    return ", ".join(f"{k.split('__')[1]}={v}" for k, v in params.items())

def optimiser(nom, espace):
    positions = {p: espace["echelles"][p].index(v) for p, v in espace["depart"].items()}
    lignes, raison, debut = [], f"{MAX_TOURS} tours atteints", time.perf_counter()
    for tour in range(1, MAX_TOURS + 1):
        debut_tour = time.perf_counter()
        grille = {p: fenetre(espace["echelles"][p], i) for p, i in positions.items()} | espace["choix"]
        recherche = GridSearchCV(espace["modele"], grille, scoring=scoreur, cv=cv, n_jobs=espace["n_jobs"], return_train_score=True, refit=False)
        recherche.fit(X_emp, y_emp, **espace.get("fit_params", {}))
        r = recherche.cv_results_
        res = pd.DataFrame({
            "tour": tour,
            "params": r["params"],
            "validation": r["mean_test_score"] * 100,
            "écart-type des plis": r["std_test_score"] * 100,
            "train": r["mean_train_score"] * 100,
        })
        res["surapprentissage (% relatif)"] = (res["train"] - res["validation"]) / res["train"] * 100
        res["admis"] = res["surapprentissage (% relatif)"] <= SURAPPRENTISSAGE_MAX
        lignes.append(res)
        admis = res[res["admis"]]
        if admis.empty:
            print(f"{nom}, tour {tour} : {len(res)} réglages, aucun admis ({round(time.perf_counter() - debut_tour)} s)", flush=True)
            raison = f"tour {tour} : aucun réglage sous {SURAPPRENTISSAGE_MAX} % de surapprentissage"
            break
        meilleur = admis.loc[admis["validation"].idxmax(), "params"]
        print(f"{nom}, tour {tour} : {len(res)} réglages, {len(admis)} admis ; meilleur admis {admis['validation'].max():.2f} % "
              f"({libelle(meilleur)}) ({round(time.perf_counter() - debut_tour)} s)", flush=True)
        nouvelles = {p: espace["echelles"][p].index(meilleur[p]) for p in positions}
        if nouvelles == positions:
            raison = f"tour {tour} : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle)"
            break
        positions = nouvelles
    historique = pd.concat(lignes, ignore_index=True)
    return historique, raison, round(time.perf_counter() - debut)

resultats = {}
for nom, espace in ESPACES.items():
    historique, raison, duree = optimiser(nom, espace)
    resultats[nom] = {"historique": historique, "raison": raison, "duree": duree}
    admis = historique[historique["admis"]]
    if admis.empty:
        print(f"{nom} : aucun réglage admis ; arrêt : {raison} ({duree} s)")
    else:
        m = admis.loc[admis["validation"].idxmax()]
        print(f"{nom} : {m['validation']:.2f} % ± {m['écart-type des plis']:.2f} ({libelle(m['params'])}) ; arrêt : {raison} ({duree} s)")

## 7. Synthèse : le meilleur réglage de chaque modèle

Pour chaque modèle, le meilleur réglage admis (surapprentissage de 7 % relatif au plus) sur l'ensemble des tours, avec son score en validation croisée, l'écart-type de ses 5 plis, son score en train et son surapprentissage. Alertes : ⚠️ variance si l'écart-type des plis dépasse 2 points ; ⚠️ aucun réglage admis.

In [ ]:
lignes, meilleurs = [], {}
for nom, r in resultats.items():
    h = r["historique"]
    admis = h[h["admis"]]
    if admis.empty:
        ligne = h.loc[h["surapprentissage (% relatif)"].idxmin()]
        alerte = "⚠️ aucun réglage admis (le moins surappris est affiché)"
    else:
        ligne = admis.loc[admis["validation"].idxmax()]
        alerte = f"⚠️ variance ({ligne['écart-type des plis']:.1f} points)" if ligne["écart-type des plis"] > VARIANCE_MAX else "—"
    meilleurs[nom] = ligne["params"]
    lignes.append({"Modèle": nom, "Réglage": libelle(ligne["params"]),
                   f"Validation ({SCORE}, %)": f"{ligne['validation']:.2f} ± {ligne['écart-type des plis']:.2f}",
                   "Train (%)": round(ligne["train"], 2),
                   "Surapprentissage (% relatif)": round(ligne["surapprentissage (% relatif)"], 2),
                   "Alerte": alerte, "Tours": int(h["tour"].max()), "Arrêt": r["raison"], "Durée (s)": r["duree"]})
display(pd.DataFrame(lignes).set_index("Modèle"))

## 7 bis. Enregistrement des modèles (`modeles/11b/`)

Chaque modèle est appris sur **tout le train** (tous les mois des clients du train, comme dans la boucle) avec son **meilleur réglage admis** (surapprentissage de 7 % relatif au plus), puis enregistré dans `modeles/11b/11b_<modèle>.pkl`, avec un fichier de synthèse `modeles/11b/11b_synthese.json` (données, corrections, variables, classes, réglage, scores de validation, durée, versions des bibliothèques). Le test de septembre n'est pas lu. Les modèles sont enregistrés avec `cloudpickle`, qui embarque aussi la fonction du logarithme : ils se rechargent avec `pickle.load`, sans redéfinir cette fonction.

In [ ]:
import json, platform, sys, time
from datetime import datetime
import catboost, sklearn
from joblib.externals import cloudpickle

dossier_modeles = Path.cwd() / "modeles" / "11b"
dossier_modeles.mkdir(parents=True, exist_ok=True)
FICHIERS = {"Régression logistique": "regression_logistique", "Arbre de décision": "arbre_de_decision",
            "RandomForest": "randomforest", "MLP": "mlp", "CatBoost": "catboost"}

synthese = {"scenario": "11b", "date": datetime.now().isoformat(timespec="seconds"), "base": "01f (mois empilés)",
            "source": "data/creditcard_pret_ingestion.csv (niveau 0), sans dpnm, sans ID, sans données personnelles",
            "corrections": "10 (codifications 1 d'avril à août) puis 09 (règle fixe des retards, facture >= 100 NT$) ; PAY_1 non corrigé",
            "cible": "PAY_1 binaire : sain (-2, -1, 0) ou en retard (2 et plus), clients dont PAY_1 n'est pas un 1",
            "variables": VARIABLES, "classes": [int(c) for c in CLASSES],
            "score": SCORE, "surapprentissage_max_pct": SURAPPRENTISSAGE_MAX,
            "versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__, "catboost": catboost.__version__, "pandas": pd.__version__},
            "modeles": {}}
for nom, params in meilleurs.items():
    debut = time.perf_counter()
    modele = clone(ESPACES[nom]["modele"]).set_params(**params).fit(X_emp, y_emp)
    fichier = dossier_modeles / f"11b_{FICHIERS[nom]}.pkl"
    with open(fichier, "wb") as f:
        cloudpickle.dump(modele, f)
    h = resultats[nom]["historique"]
    ligne = h[h["params"].map(lambda p: p == params)].iloc[-1]
    synthese["modeles"][nom] = {"fichier": fichier.name, "reglage": {k.split("__")[1]: (v if isinstance(v, (int, float, str)) or v is None else str(v)) for k, v in params.items()},
                                "validation_pct": round(float(ligne["validation"]), 2), "ecart_type_plis_pct": round(float(ligne["écart-type des plis"]), 2),
                                "train_pct": round(float(ligne["train"]), 2), "surapprentissage_relatif_pct": round(float(ligne["surapprentissage (% relatif)"]), 2),
                                "duree_apprentissage_s": round(time.perf_counter() - debut)}
    print(f"{nom} : enregistré dans {fichier.name}", flush=True)
(dossier_modeles / "11b_synthese.json").write_text(json.dumps(synthese, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"Synthèse : {dossier_modeles / '11b_synthese.json'}")

**Les variables annulées par la régression logistique retenue** : elle est réapprise sur tous les exemples (tous les mois) avec son meilleur réglage ; pour chaque variable (colonnes par mois), le nombre de ses coefficients (un par classe, et un par valeur pour les codifications) et la part mise à zéro. Avec une pénalité L2 (`l1_ratio` = 0), aucun coefficient n'est annulé.

In [ ]:
lr = clone(ESPACES["Régression logistique"]["modele"]).set_params(**meilleurs["Régression logistique"]).fit(X_emp, y_emp)
noms = lr[1].get_feature_names_out()   # colonnes après le prétraitement (codifications en une colonne par valeur)
variables = [n.split("__", 1)[1].rsplit("_", 1)[0] if n.startswith("categories__") else n.split("__", 1)[1] for n in noms]
coefficients = pd.DataFrame(lr[-1].coef_, columns=variables)
annulation = pd.DataFrame({"Coefficients": coefficients.shape[0] * pd.Series(variables).value_counts(),
                           "Mis à zéro": (coefficients == 0).sum().groupby(level=0).sum()}).reindex(VARIABLES)
annulation["Part mise à zéro (%)"] = (annulation["Mis à zéro"] / annulation["Coefficients"] * 100).round(1)
print(f"Réglage : {libelle(meilleurs['Régression logistique'])}")
display(annulation)

## 8. Les modèles réglés, sur le septembre du train (prédictions hors pli)

Chaque modèle, avec son meilleur réglage, apprend sur chaque pli (tous les mois des autres clients) et prédit le septembre des clients du pli. Chaque client du train de septembre est ainsi prédit une fois, par un modèle qui ne l'a jamais vu, à aucun mois. Les tableaux sont ceux du 01e, sur les mêmes 21 049 clients : ils se comparent directement.

In [ ]:
X_sept, y_sept = X_emp.iloc[sept], y_emp.iloc[sept]
rang = np.full(len(emp), -1)
rang[sept] = np.arange(len(sept))   # position de chaque exemple de septembre dans X_sept

predictions_hors_pli = {}
for nom, params in meilleurs.items():
    p = np.empty(len(sept), dtype=int)
    for apprentissage, validation in cv:
        m = clone(ESPACES[nom]["modele"]).set_params(**params).fit(X_emp.iloc[apprentissage], y_emp.iloc[apprentissage])
        p[rang[validation]] = np.asarray(m.predict(X_emp.iloc[validation])).ravel().astype(int)
    predictions_hors_pli[nom] = p
    print(f"{nom} : fait")
predictions_hors_pli["Règle « même codification qu'en août »"] = X_sept["codif_m1"].to_numpy().astype(int)
predictions_hors_pli["Toujours 0"] = np.zeros(len(X_sept), dtype=int)

lignes = []
for nom, p in predictions_hors_pli.items():
    b = bonnes_predictions(y_sept, p)
    lignes.append({"Modèle": nom, "Bonnes prédictions (%)": round(b["global"], 1),
                   **{f"Famille {NOMS_FAMILLES[f]} (%)": round(b[f], 1) for f in FAMILLES}})
display(pd.DataFrame(lignes).set_index("Modèle"))

**Comment lire** : une ligne par modèle réglé, sur les prédictions hors pli du train ; les deux dernières lignes sont les points de comparaison. Mêmes mesures et mêmes clients que dans le 01e.

In [ ]:
famille_aout, famille_sept = famille_pay(X_sept["codif_m1"]), famille_pay(y_sept)
change = famille_aout != famille_sept
TYPES_CHANGEMENT = {
    "Sortie du retard (2 et plus en août, sain en septembre)": (famille_aout == 2) & (famille_sept != 2),
    "Retard maintenu (2 et plus en août et en septembre ; pas un changement)": (famille_aout == 2) & (famille_sept == 2),
    "Nouveau retard (sain en août, 2 et plus en septembre)": (famille_aout != 2) & (famille_sept == 2),
    "Tous les clients qui changent": change,
}
lignes = []
for type_changement, masque in TYPES_CHANGEMENT.items():
    resultat = {"Type de changement": type_changement, "Clients": int(masque.sum())}
    for nom, p in predictions_hors_pli.items():
        if nom == "Toujours 0":
            continue
        resultat[f"{nom} : retrouvés (%)"] = round((famille_pay(p)[masque] == famille_sept[masque]).mean() * 100, 1)
    lignes.append(resultat)
display(pd.DataFrame(lignes).set_index("Type de changement"))

**Comment lire** : comme dans le 01e, une ligne par type de changement entre août et septembre, avec la part des clients rangés dans leur vraie famille de septembre ; la règle naïve est à 0 % sur les changements et à 100 % sur les retards maintenus.

## 9. L'importance des variables

**Importance par permutation** : pour chaque variable, ses valeurs sont mélangées entre les clients de la validation (leur septembre), et l'on mesure **la baisse du score** (moyenne des familles, en points). Une variable dont le mélange ne fait rien perdre n'est pas utilisée par le modèle. Le calcul se fait pli par pli (mêmes 5 plis, 5 mélanges par variable), puis la moyenne des plis est affichée.

Le second tableau regroupe les variables **par ancienneté** : le mois de la codification (M : `solde_m0`, `paiement_m0`), le mois précédent (M-1 : `codif_m1`, `solde_m1`, `paiement_m1`), et ainsi de suite jusqu'à M-5, puis le plafond. Pour septembre, M est septembre et M-5 est avril.

In [ ]:
from sklearn.inspection import permutation_importance

def anciennete(variable):
    return "plafond" if variable == "LIMIT_BAL" else ("M" if variable.endswith("m0") else f"M-{variable[-1]}")

importances = {}
for nom, params in meilleurs.items():
    par_pli = []
    for apprentissage, validation in cv:
        m = clone(ESPACES[nom]["modele"]).set_params(**params).fit(X_emp.iloc[apprentissage], y_emp.iloc[apprentissage])
        r = permutation_importance(m, X_emp.iloc[validation], y_emp.iloc[validation], scoring=scoreur,
                                   n_repeats=5, random_state=42, n_jobs=-1)
        par_pli.append(r.importances_mean * 100)
    importances[nom] = pd.Series(np.mean(par_pli, axis=0), index=X_emp.columns)
    print(f"{nom} : fait")

table = pd.DataFrame(importances)
table["Moyenne des modèles"] = table.mean(axis=1)
display(Markdown("**Par variable** (baisse du score en points, de la plus importante à la moins importante en moyenne)"))
display(table.sort_values("Moyenne des modèles", ascending=False).round(2))
par_anciennete = table.groupby(table.index.map(anciennete)).sum()
display(Markdown("**Par ancienneté** (somme des variables de chaque mois)"))
display(par_anciennete.reindex(["M", "M-1", "M-2", "M-3", "M-4", "M-5", "plafond"]).round(2))

**Comment lire** : une ligne par variable (ou par ancienneté), une colonne par modèle ; la valeur est la baisse du score, en points, quand la variable est mélangée. Proche de 0 : le modèle ne s'en sert pas ; négative : bruit. Si M-3 à M-5 sont proches de 0 pour tous les modèles, les mois anciens n'apportent rien.